# SmartCare Hospital AI Dataset — CCS3440 Artificial Intelligence Coursework
## Option C — Disease Risk Classification (`disease_risk_level`: Low / Medium / High)

## Task 02 — Dataset Understanding

### Purpose

This task is used to understand the SmartCare Hospital AI dataset before data
preprocessing and model development.

In this task:

- The dataset and data dictionary are loaded.
- The number of rows and columns is checked.
- Input and target variables are identified.
- Numerical and categorical attributes are explored.
- Missing values and duplicate records are checked.
- Data-quality issues are identified.
- The distribution of the target classes is examined.

The original dataset is stored in `df_raw`. A copy named `df` is used for the analysis
to keep the original dataset unchanged.


# **Task 2 - Dataset Understanding**

Goal: understand what the dataset contains, what each column means, which columns are candidate inputs vs. targets for Option C, and what data-quality issues exist before any cleaning is attempted (cleaning itself happens in Task 03).





## 2.1 Load the data and the data dictionary

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split

pd.set_option('display.max_columns', None)
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (8, 5)
RANDOM_STATE = 42

df_raw = pd.read_csv('/content/drive/MyDrive/SmartCare/smartcare_ai_dataset_1000.csv')
data_dict = pd.read_csv('/content/drive/MyDrive/SmartCare/smartcare_ai_dataset_data_dictionary.csv')
df = df_raw.copy()   # all transformations happen on this working copy; df_raw stays untouched for viva reference
df.shape

print(f"Records: {df.shape[0]}, Columns: {df.shape[1]}")
df.head()

Records: 1000, Columns: 33


,record_id,patient_id,age,gender,blood_group,department,diagnosis,appointment_date,waiting_days,previous_appointments,missed_previous_appointments,appointment_status,admitted,room_type,length_of_stay_days,previous_admissions,systolic_bp,diastolic_bp,blood_sugar_mg_dl,cholesterol_mg_dl,bmi,lab_tests_count,treatments_count,consultation_fee_lkr,room_charge_lkr,lab_charge_lkr,medicine_charge_lkr,total_bill_lkr,payment_status,payment_method,no_show,readmitted_30_days,disease_risk_level
0,1,P10001,53,Male,A-,General Medicine,Migraine,2025-04-10,10,1,0,Completed,0,NaN,0,1,127,75,117,211,26.1,0,3,2000,0,0,11596,13596,Paid,Insurance,0,0,High
1,2,P10002,26,Male,B-,General Medicine,Diabetes,2025-05-15,2,3,1,Completed,0,NaN,0,0,130,73,136,173,32.8,0,1,2000,0,0,3652,5652,Paid,Insurance,0,0,Medium
2,3,P10003,22,Male,B+,Orthopedics,Back Pain,2025-07-09,22,7,1,No-Show,0,NaN,0,1,141,64,90,176,29.4,1,0,2500,0,1200,2562,6262,Unpaid,Insurance,1,0,Medium
3,4,P10004,44,Female,AB-,Cardiology,Asthma,2025-10-16,16,1,0,Completed,0,NaN,0,0,124,82,126,189,24.9,2,1,2000,0,5000,10262,17262,Paid,Online,0,0,Medium
4,5,P10005,51,Female,O+,Neurology,Hypertension,2025-12-18,12,4,0,Scheduled,0,NaN,0,1,119,81,65,195,27.0,2,0,4000,0,6000,10414,20414,Paid,Cash,0,0,Medium


In [ ]:
# Data dictionary — the authoritative description of every column
pd.set_option('display.max_rows', None)
data_dict

,Column,Description
0,record_id,Unique row identifier
1,patient_id,Synthetic patient identifier
2,age,Patient age in years
3,gender,Patient gender
4,blood_group,Patient blood group
5,department,Hospital department
6,diagnosis,Primary diagnosis category
7,appointment_date,Appointment date
8,waiting_days,Number of days between booking and appointment
9,previous_appointments,Number of previous appointments


## 2.2 Structural overview

`df.info()` gives dtypes and non-null counts in one pass, which is the fastest way to
spot columns that are the wrong type (e.g. dates stored as text) or that already show
missing values.

In [ ]:
df.info()



<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 33 columns):
 #   Column                        Non-Null Count  Dtype  
---  ------                        --------------  -----  
 0   record_id                     1000 non-null   int64  
 1   patient_id                    1000 non-null   object 
 2   age                           1000 non-null   int64  
 3   gender                        1000 non-null   object 
 4   blood_group                   1000 non-null   object 
 5   department                    1000 non-null   object 
 6   diagnosis                     1000 non-null   object 
 7   appointment_date              1000 non-null   object 
 8   waiting_days                  1000 non-null   int64  
 9   previous_appointments         1000 non-null   int64  
 10  missed_previous_appointments  1000 non-null   int64  
 11  appointment_status            1000 non-null   object 
 12  admitted                      1000 non-null   int64  
 13  room

## 2.3 Grouping columns by role (per the coursework brief)

The dataset mixes four themes — patient info, clinical info, hospital operations, and
financial data — plus three possible AI targets. Only one target is used per option;
the other two are dropped later (Task 03) to avoid using one label to predict another.

In [ ]:
# Group dataset columns by category and check whether all expected columns exist.
column_groups = {
    'Identifiers':            ['record_id', 'patient_id'],
    'Patient info':            ['age', 'gender', 'blood_group'],
    'Clinical info':           ['diagnosis', 'systolic_bp', 'diastolic_bp',
                                 'blood_sugar_mg_dl', 'cholesterol_mg_dl', 'bmi'],
    'Appointment / operations': ['department', 'appointment_date', 'waiting_days',
                                 'previous_appointments', 'missed_previous_appointments',
                                 'appointment_status', 'admitted', 'room_type',
                                 'length_of_stay_days', 'previous_admissions',
                                 'lab_tests_count', 'treatments_count'],
    'Financial':               ['consultation_fee_lkr', 'room_charge_lkr', 'lab_charge_lkr',
                                 'medicine_charge_lkr', 'total_bill_lkr',
                                 'payment_status', 'payment_method'],
    'AI targets (pick ONE)':   ['no_show', 'readmitted_30_days', 'disease_risk_level'],
}

for group, cols in column_groups.items():
    missing_from_df = [c for c in cols if c not in df.columns]
    print(f"{group:28s} ({len(cols)} cols): {cols}")
    if missing_from_df:
        print(f"  !! not found in dataframe: {missing_from_df}")



Identifiers                  (2 cols): ['record_id', 'patient_id']
Patient info                 (3 cols): ['age', 'gender', 'blood_group']
Clinical info                (6 cols): ['diagnosis', 'systolic_bp', 'diastolic_bp', 'blood_sugar_mg_dl', 'cholesterol_mg_dl', 'bmi']
Appointment / operations     (12 cols): ['department', 'appointment_date', 'waiting_days', 'previous_appointments', 'missed_previous_appointments', 'appointment_status', 'admitted', 'room_type', 'length_of_stay_days', 'previous_admissions', 'lab_tests_count', 'treatments_count']
Financial                    (7 cols): ['consultation_fee_lkr', 'room_charge_lkr', 'lab_charge_lkr', 'medicine_charge_lkr', 'total_bill_lkr', 'payment_status', 'payment_method']
AI targets (pick ONE)        (3 cols): ['no_show', 'readmitted_30_days', 'disease_risk_level']


## 2.4 Selected target for this notebook

Target : `disease_risk_level` — a 3-class label (Low / Medium / High), so this is a
multi-class classification problem (Option C). `no_show` and `readmitted_30_days`
remain in the raw data but are treated purely as other AI targets — they are dropped
from the feature set in Task 03 rather than used as predictors, since a real deployment
would not have next-visit outcomes available at prediction time for this task.


In [ ]:
# Display the count and percentage distribution of each disease risk class.
target_col = 'disease_risk_level'

print("Class counts:")
print(df[target_col].value_counts())
print("\nClass proportions (%):")
print((df[target_col].value_counts(normalize=True) * 100).round(1))

Class counts:
disease_risk_level
Medium    469
High      400
Low       131
Name: count, dtype: int64

Class proportions (%):
disease_risk_level
Medium    46.9
High      40.0
Low       13.1
Name: proportion, dtype: float64


**Insights:** the classes are imbalanced Medium (~ 47%) and High (~ 40%) dominate, while
Low is a minority class (~13%). This is flagged now because it drives two later
decisions: a stratified train/test split (Task 03) and the choice to report
macro-averaged metrics, not just accuracy, in Task 06.

## 2.5 Display Summary Statistics

In [ ]:
numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
df[numeric_cols].describe().T.round(2)

,count,mean,std,min,25%,50%,75%,max
record_id,1000.0,500.50,288.82,1.0,250.75,500.5,750.25,1000.0
age,1000.0,44.74,17.85,1.0,33.00,44.0,57.00,90.0
waiting_days,1000.0,21.85,13.04,0.0,11.00,22.0,34.00,44.0
previous_appointments,1000.0,2.88,1.69,0.0,2.00,3.0,4.00,10.0
missed_previous_appointments,1000.0,0.55,0.74,0.0,0.00,0.0,1.00,4.0
admitted,1000.0,0.33,0.47,0.0,0.00,0.0,1.00,1.0
length_of_stay_days,1000.0,1.10,1.89,0.0,0.00,0.0,2.00,9.0
previous_admissions,1000.0,0.86,0.96,0.0,0.00,1.0,1.00,5.0
systolic_bp,1000.0,128.42,15.49,85.0,117.00,128.0,139.00,178.0
diastolic_bp,1000.0,78.83,10.04,50.0,72.00,79.0,86.00,111.0



- `age` (18–~ 85), `bmi` (~ 14–39), blood pressure and cholesterol ranges all look
  clinically plausible.
- `waiting_days`, charges and `length_of_stay_days` are right-skewed (mean > median),
  which is expected for cost/time data and is revisited with histograms in Task 04.


## 2.6 Load and Display the Data Dictionary



In [ ]:
# Display unique value counts for each categorical column, excluding appointment_date.
categorical_cols = df.select_dtypes(include=['object']).columns.tolist()
categorical_cols = [c for c in categorical_cols if c != 'appointment_date']

for col in categorical_cols:
    print(f"--- {col} ({df[col].nunique()} unique) ---")
    print(df[col].value_counts(dropna=False))
    print()

--- patient_id (1000 unique) ---
patient_id
P11000    1
P10001    1
P10002    1
P10003    1
P10004    1
P10005    1
P10006    1
P10007    1
P10984    1
P10983    1
P10982    1
P10981    1
P10980    1
P10979    1
P10978    1
P10977    1
P10976    1
P10975    1
P10974    1
P10973    1
P10972    1
P10971    1
P10970    1
P10969    1
P10040    1
P10039    1
P10038    1
P10037    1
P10036    1
P10035    1
P10034    1
P10033    1
P10032    1
P10031    1
P10030    1
P10029    1
P10028    1
P10027    1
P10026    1
P10025    1
P10056    1
P10055    1
P10054    1
P10053    1
P10052    1
P10051    1
P10050    1
P10049    1
P10048    1
P10047    1
P10046    1
P10045    1
P10044    1
P10043    1
P10042    1
P10041    1
P10072    1
P10071    1
P10070    1
P10069    1
P10068    1
P10067    1
P10066    1
P10065    1
P10064    1
P10063    1
P10062    1
P10061    1
P10060    1
P10059    1
P10058    1
P10057    1
P10088    1
P10087    1
P10086    1
P10085    1
P10084    1
P10083    1
P10082    1
P10081  

## 2.7 Data-quality issues found (addressed in Task 03)

Inspecting the raw values above and cross-checking logically related columns surfaces
four concrete issues, each carried forward and fixed with justification in Task 03:

| # | Issue | Column(s) | Rows affected | Type |
|---|---|---|---|---|
| 1 | Missing values | `room_type` | 906 / 1000 (90.6%) | Structural (mostly not-admitted patients) |
| 2 | Logically impossible count | `missed_previous_appointments > previous_appointments` | small | Data entry / generation error |
| 3 | Physiologically invalid reading | `systolic_bp <= diastolic_bp` | small | Likely transposition error |
| 4 | Billing inconsistency | `treatments_count == 0` but `medicine_charge_lkr > 0` | small | Recording inconsistency (out of scope for this target) |

The next two cells quantify #1–#3 precisely; #4 is quantified in Task 03 where it is
also resolved.


In [ ]:
# Issue 1: missing values, and whether they concentrate somewhere specific
missing = df.isnull().sum()
missing = missing[missing > 0]
print("Columns with missing values:")
print(missing)

print("\nroom_type missingness by admission status (is it structural?):")
print(df.groupby('admitted')['room_type'].apply(lambda s: s.isnull().sum()))


Columns with missing values:
room_type    906
dtype: int64

room_type missingness by admission status (is it structural?):
admitted
0    670
1    236
Name: room_type, dtype: int64


**Observation**: The classes are imbalanced (Medium ≈ 47%, High ≈ 40%, Low ≈ 13%). We account for this later using class_weight='balanced' where supported, stratified train/test splitting, and by reporting macro-averaged (not just accuracy) metrics.



In [ ]:
# Issues 2 & 3: logical / physiological consistency checks
bad_missed = (df['missed_previous_appointments'] > df['previous_appointments']).sum()
bad_bp = (df['systolic_bp'] <= df['diastolic_bp']).sum()
billing_inconsistent = ((df['treatments_count'] == 0) & (df['medicine_charge_lkr'] > 0)).sum()

print(f"missed_previous_appointments > previous_appointments : {bad_missed} rows")
print(f"systolic_bp <= diastolic_bp                          : {bad_bp} rows")
print(f"treatments_count == 0 but medicine_charge_lkr > 0     : {billing_inconsistent} rows")


missed_previous_appointments > previous_appointments : 16 rows
systolic_bp <= diastolic_bp                          : 3 rows
treatments_count == 0 but medicine_charge_lkr > 0     : 212 rows


## 2.8 Duplicate check (first pass)


A full check (row-level, record_id, and patient_id) is repeated formally in Task 03; here it is a quick first look to complete the dataset-understanding picture.

In [ ]:
# Check for fully duplicated records, duplicate record IDs, and repeated patient IDs.
print(f"Fully duplicated rows : {df.duplicated().sum()}")
print(f"Duplicate record_id   : {df['record_id'].duplicated().sum()}")
print(f"Repeated patient_id   : {df['patient_id'].duplicated().sum()} "
      f"(expected — a patient can have multiple visits, this is NOT an error)")

Fully duplicated rows : 0
Duplicate record_id   : 0
Repeated patient_id   : 0 (expected — a patient can have multiple visits, this is NOT an error)


## 2.9 Task 02 summary

- 1000 records, 33 columns, no full-row duplicates, one column (`room_type`) with
  genuine missing data plus three small logical inconsistencies.
- Target: `disease_risk_level` (Low/Medium/High) — imbalanced multi-class label.
- Candidate predictors: patient demographics, clinical vitals, and operational
  counts. `total_bill_lkr` and its components are financial outcomes of a visit
  rather than pre-visit risk indicators, so their usefulness as predictors is checked
  with correlation analysis in Task 03/04 rather than assumed.
- Excluded from features regardless of correlation: `record_id`, `patient_id`
  (identifiers only) and `no_show`, `readmitted_30_days` (other AI targets — using one
  target to predict another would be data leakage / poor methodology).

In [ ]:
import os

print("Files in Colab:")

for file_name in os.listdir('/content'):
    print(file_name)

Files in Colab:
.config
smartcare_ai_dataset_data_dictionary (1).csv
smartcare_ai_dataset_1000 (1).csv
sample_data
